# Station 01｜CNN：動物之家照片智慧分流

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/day6_ai_solution_lab/01_cnn_pet_router.ipynb)

**客戶任務：** 高信心的貓狗照片自動分流；低信心或不確定照片交由志工確認。

- Kaggle 教學資料：[Cats and Dogs image classification](https://www.kaggle.com/datasets/samuelcortinhas/cats-and-dogs-image-classification)（CC0，約千張影像）
- 經典題目：[Dogs vs. Cats Competition](https://www.kaggle.com/competitions/dogs-vs-cats/data)
- 30 分鐘節奏：5 分鐘讀情境 → 8 分鐘跑模型 → 12 分鐘改門檻 → 5 分鐘記錄。
- 本站只做**預測與拒絕預測門檻**；模型信心不等於正確，也不能辨識所有非貓狗圖片。


## 1. Setup 與實驗設定


In [ ]:
%pip -q install kagglehub==1.0.2 gradio==6.20.0


In [ ]:
from pathlib import Path
import random
import re
import time

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

SEED = 20260719
QUICK_MODE = True
IMAGE_SIZE = (160, 160)
MAX_IMAGES = 800 if QUICK_MODE else 2_000
BASELINE_EPOCHS = 1 if QUICK_MODE else 4
CANDIDATE_EPOCHS = 1 if QUICK_MODE else 4

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "CPU fallback")


## 2. 從 KaggleHub 下載公開教學資料

程式會從資料夾名稱或檔名判斷 `cat`／`dog`，不假設固定壓縮路徑。


In [ ]:
DATASET_HANDLE = "samuelcortinhas/cats-and-dogs-image-classification"
data_root = Path(kagglehub.dataset_download(DATASET_HANDLE))

image_paths = sorted(
    path for path in data_root.rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

def infer_label(path):
    for part in reversed(path.parts[:-1]):
        key = part.lower().strip(" _-")
        if key in {"cat", "cats"}:
            return 0
        if key in {"dog", "dogs"}:
            return 1
    stem = path.stem.lower()
    if re.match(r"^cat(?:[._ -]|$)", stem):
        return 0
    if re.match(r"^dog(?:[._ -]|$)", stem):
        return 1
    return None

records = [(str(path), infer_label(path)) for path in image_paths]
frame = pd.DataFrame(records, columns=["path", "label"]).dropna()
frame["label"] = frame["label"].astype(int)
if frame["label"].nunique() != 2:
    raise RuntimeError("無法辨認貓狗資料夾；請把完整錯誤訊息交給講師。")

if len(frame) > MAX_IMAGES:
    frame = (
        frame.groupby("label", group_keys=False)
        .apply(lambda part: part.sample(min(len(part), MAX_IMAGES // 2), random_state=SEED))
        .reset_index(drop=True)
    )

train_frame, test_frame = train_test_split(
    frame, test_size=0.25, stratify=frame["label"], random_state=SEED
)
print("Data root:", data_root)
print("Train/Test:", len(train_frame), len(test_frame))
print(frame["label"].map({0: "cat", 1: "dog"}).value_counts())


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)

def make_dataset(part, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((part["path"].values, part["label"].values))
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.shuffle(len(part), seed=SEED)
    return dataset.batch(32).prefetch(AUTOTUNE)

train_ds = make_dataset(train_frame, training=True)
test_ds = make_dataset(test_frame)


## 3. Baseline：小型 CNN

Baseline 的任務是提供可比較起點，不追求排行榜成績。


In [ ]:
baseline = tf.keras.Sequential([
    tf.keras.layers.Input((*IMAGE_SIZE, 3)),
    tf.keras.layers.Rescaling(1 / 255),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="small_cnn_baseline")
baseline.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
baseline.fit(train_ds, validation_data=test_ds, epochs=BASELINE_EPOCHS, verbose=2)
baseline_prob = baseline.predict(test_ds, verbose=0).ravel()


## 4. Candidate：MobileNetV2 遷移學習

CPU 備援仍可跑；若教室網路無法下載 ImageNet 權重，保留 Baseline 結果並跳到門檻實驗。


In [ ]:
candidate_available = True
try:
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.05, seed=SEED),
    ])
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3), include_top=False, weights="imagenet"
    )
    backbone.trainable = False
    inputs = tf.keras.Input((*IMAGE_SIZE, 3))
    x = augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    candidate = tf.keras.Model(inputs, outputs, name="mobilenetv2_candidate")
    candidate.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    candidate.fit(train_ds, validation_data=test_ds, epochs=CANDIDATE_EPOCHS, verbose=2)
    candidate_prob = candidate.predict(test_ds, verbose=0).ravel()
except Exception as exc:
    candidate_available = False
    candidate = baseline
    candidate_prob = baseline_prob
    print("Candidate 暫時不可用，改用 Baseline 完成門檻實驗：", type(exc).__name__, exc)


## 5. 必做實驗：改「人工覆核門檻」

`REVIEW_THRESHOLD` 代表模型信心低於多少時，不自動分流。請只改這一個值，再執行本格與下一格。


In [ ]:
REVIEW_THRESHOLD = 0.75  # TODO(學員必改)：比較 0.60、0.75 或 0.85

y_true = test_frame["label"].to_numpy()

def evaluate_with_review(probabilities, review_threshold):
    predicted = (probabilities >= 0.5).astype(int)
    confidence = np.maximum(probabilities, 1 - probabilities)
    auto_mask = confidence >= review_threshold
    return {
        "accuracy": accuracy_score(y_true, predicted),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "f1": f1_score(y_true, predicted, zero_division=0),
        "coverage": auto_mask.mean(),
        "review_rate": 1 - auto_mask.mean(),
        "high_conf_accuracy": accuracy_score(y_true[auto_mask], predicted[auto_mask]) if auto_mask.any() else np.nan,
    }

comparison = pd.DataFrame({
    "Baseline": evaluate_with_review(baseline_prob, REVIEW_THRESHOLD),
    "Candidate": evaluate_with_review(candidate_prob, REVIEW_THRESHOLD),
}).T
display(comparison.round(3))


In [ ]:
predicted = (candidate_prob >= 0.5).astype(int)
confidence = np.maximum(candidate_prob, 1 - candidate_prob)
errors = test_frame.copy()
errors["actual"] = y_true
errors["predicted"] = predicted
errors["confidence"] = confidence
errors = errors[errors["actual"] != errors["predicted"]].sort_values("confidence", ascending=False)

if len(errors):
    fig, axes = plt.subplots(1, min(3, len(errors)), figsize=(10, 3))
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, errors.head(3).iterrows()):
        ax.imshow(plt.imread(row["path"]))
        ax.set_title(f"actual={row.actual}, pred={row.predicted}\nconf={row.confidence:.2f}")
        ax.axis("off")
    plt.tight_layout()
else:
    print("本次小樣本沒有錯分；請降低訓練量或另找邊界案例。")


## 6. 課堂暫時 Demo（選用）


In [ ]:
import gradio as gr
from PIL import Image

MODEL_VERSION = "station01-candidate-v1" if candidate_available else "station01-baseline-v1"

def predict_pet(image):
    if image is None:
        return {"status": "請先上傳圖片", "model_version": MODEL_VERSION}
    array = np.asarray(image.convert("RGB").resize(IMAGE_SIZE), dtype=np.float32)
    probability_dog = float(candidate.predict(array[None, ...], verbose=0)[0, 0])
    label = "狗" if probability_dog >= 0.5 else "貓"
    confidence_value = max(probability_dog, 1 - probability_dog)
    return {
        "prediction": label,
        "confidence": round(confidence_value, 3),
        "action": "自動分流" if confidence_value >= REVIEW_THRESHOLD else "建議人工確認",
        "model_version": MODEL_VERSION,
    }

demo = gr.Interface(
    fn=predict_pet,
    inputs=gr.Image(type="pil", label="上傳貓／狗照片"),
    outputs=gr.JSON(label="分流建議"),
    title="動物之家照片分流（課堂暫時 Demo）",
    description="非貓狗影像也可能得到高信心；正式系統需要 OOD 檢查。",
)
print("需要介面時再執行：demo.launch(share=True)")


## 7. 下載迷你實驗卡


In [ ]:
from pathlib import Path

experiment_card = f'''# CNN 站迷你實驗卡

- 組別：請填寫
- 我修改了：REVIEW_THRESHOLD = {REVIEW_THRESHOLD}
- 原本結果：請貼上修改前的 Coverage / F1
- 修改後結果：請貼上修改後的 Coverage / F1
- 可能原因：請填寫
- 最大失敗情境：請填寫
- Seed：{SEED}
- 模型版本：{MODEL_VERSION}
- 限制：二元分類信心不等於能辨認領域外圖片。
'''
output_path = Path("/content/station01_cnn_experiment_card.md")
output_path.write_text(experiment_card, encoding="utf-8")
print(experiment_card)
print("Saved:", output_path)
